In [1]:
import numpy as np
import cv2
import math

import plotly.graph_objects as go
import plotly.express as px

from IPython.display import Image, display, HTML

In [3]:
coords_left = []

# Mouse callback for left image
def mouse_callback_left(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        if len(coords_left) < 72:
            coords_left.append([x,y,1])
            color = (255,255,255)
            cv2.circle(img_left, (x,y), 7, color, -1)
            cv2.imshow("LeftCameraScene", img_left)
        

img_left = cv2.imread("left-camera.jpeg")
cv2.imshow("LeftCameraScene", img_left)
cv2.setMouseCallback("LeftCameraScene", mouse_callback_left)

cv2.waitKey(0)
cv2.destroyAllWindows()
cv2.waitKey(1)


-1

In [5]:
coords_right = []

# Mouse callback for right image
def mouse_callback_right(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        if len(coords_right) < 72:
            coords_right.append([x,y,1])
            color = (255,255,255)
            cv2.circle(img_right, (x,y), 7, color, -1)
            cv2.imshow("RightCameraScene", img_right)

img_right = cv2.imread("right-camera.jpeg")
cv2.imshow("RightCameraScene", img_right)
cv2.setMouseCallback("RightCameraScene", mouse_callback_right)

cv2.waitKey(0)
cv2.destroyAllWindows()
cv2.waitKey(1)


-1

In [6]:
def afinize(vertex):
    vertex = vertex/vertex[2]
    return np.array([vertex[0], vertex[1]])

def homogeneous(vertex):
    vertex.append(1)

def eighth_vertex(vertices):
    x1 = np.cross(np.cross(vertices[1], vertices[4]), np.cross(vertices[2], vertices[5]))
    x2 = np.cross(np.cross(vertices[0], vertices[3]), np.cross(vertices[4], vertices[1]))
    x3 = np.cross(np.cross(vertices[0], vertices[3]), np.cross(vertices[2], vertices[5]))
    
    x1 = afinize(x1)
    x2 = afinize(x2)
    x3 = afinize(x3)
    x = list(1/3 * (x1 + x2 + x3))
    homogeneous(x)

    y1 = np.cross(np.cross(vertices[3], vertices[4]), np.cross(vertices[1], vertices[0]))
    y2 = np.cross(np.cross(vertices[3], vertices[4]), np.cross(vertices[5], vertices[6]))
    y3 = np.cross(np.cross(vertices[5], vertices[6]), np.cross(vertices[1], vertices[0]))

    y1 = afinize(y1)
    y2 = afinize(y2)
    y3 = afinize(y3)
    y = list(1/3 * (y1 + y2 + y3))
    homogeneous(y)

    missing = np.cross(np.cross(x, vertices[6]), np.cross(vertices[2], y))
    
    return list(np.round(missing/missing[2]))

def missing_vertex(vertices):
    x = np.cross(np.cross(vertices[0], vertices[1]), np.cross(vertices[3], vertices[4]))
    y = np.cross(np.cross(vertices[2], vertices[1]), np.cross(vertices[5], vertices[4]))

    missing = np.cross(np.cross(x,vertices[2]), np.cross(y, vertices[0]))

    return list(np.round(missing/missing[2]))

In [144]:
B_l = np.array(coords_left[:7])
A_l = np.array(coords_left[7:14])
T_l = np.array(coords_left[14:18])
R_l = np.array(coords_left[18:25])
W_l = np.array(coords_left[25:52])
S_l = np.array(coords_left[52:72])

In [145]:
B_r = np.array(coords_right[:7])
A_r = np.array(coords_right[7:14])
T_r = np.array(coords_right[14:18])
R_r = np.array(coords_right[18:25])
W_r = np.array(coords_right[25:52])
S_r = np.array(coords_right[52:72])

In [146]:
b1_l,b2_l,b3_l,b4_l,b5_l,b6_l,b7_l = B_l
a1_l,a2_l,a3_l,a4_l,a5_l,a6_l,a7_l = A_l
t1_l, t2_l, t3_l, t4_l = T_l
r1_l,r2_l,r3_l,r4_l,r5_l,r6_l,r7_l = R_l
P1_l, P2_l, P3_l, P_l = W_l[:7], W_l[7:14], W_l[14:21], W_l[21:27]
p11_l, p12_l, p13_l, p14_l, p15_l, p16_l, p17_l = P1_l
p21_l, p22_l, p23_l, p24_l, p25_l, p26_l, p27_l = P2_l
p31_l, p32_l, p33_l, p34_l, p35_l, p36_l, p37_l = P3_l
p1_l, p2_l, p3_l, p4_l, p5_l, p6_l = P_l
s1_l,s2_l,s3_l,s4_l,s5_l,s6_l,s7_l,s8_l,s9_l,s10_l,s11_l,s12_l,s13_l,s14_l,s15_l,s16_l,s17_l,s18_l,s19_l,s20_l = S_l


In [147]:
b1_r,b2_r,b3_r,b4_r,b5_r,b6_r,b7_r = B_r
a1_r,a2_r,a3_r,a4_r,a5_r,a6_r,a7_r = A_r
t1_r, t2_r, t3_r, t4_r = T_r
r1_r,r2_r,r3_r,r4_r,r5_r,r6_r,r7_r = R_r
P1_r, P2_r, P3_r, P_r = W_r[:7], W_r[7:14], W_r[14:21], W_r[21:27]
p11_r, p12_r, p13_r, p14_r, p15_r, p16_r, p17_r = P1_r
p21_r, p22_r, p23_r, p24_r, p25_r, p26_r, p27_r = P2_r
p31_r, p32_r, p33_r, p34_r, p35_r, p36_r, p37_r = P3_r
p1_r, p2_r, p3_r, p4_r, p5_r, p6_r = P_r
s1_r,s2_r,s3_r,s4_r,s5_r,s6_r,s7_r,s8_r,s9_r,s10_r,s11_r,s12_r,s13_r,s14_r,s15_r,s16_r,s17_r,s18_r,s19_r,s20_r = S_r

In [150]:
b8_l = np.array(list(map(int,eighth_vertex([b1_l,b2_l,b3_l,b4_l,b5_l,b6_l,b7_l]))))
b8_r = np.array(list(map(int,eighth_vertex([b1_r,b2_r,b3_r,b4_r,b5_r,b6_r,b7_r]))))
B_l = np.vstack([B_l, b8_l])
B_r = np.vstack([B_r, b8_r])

a8_l = np.array(list(map(int,eighth_vertex([a1_l,a2_l,a3_l,a4_l,a5_l,a6_l,a7_l]))))
a8_r = np.array(list(map(int,eighth_vertex([a1_r,a2_r,a3_r,a4_r,a5_r,a6_r,a7_r]))))
A_l = np.vstack([A_l, a8_l])
A_r = np.vstack([A_r, a8_r])

r8_l = np.array(list(map(int,eighth_vertex([r1_l,r2_l,r3_l,r4_l,r5_l,r6_l,r7_l]))))
r8_r = np.array(list(map(int,eighth_vertex([r1_r,r2_r,r3_r,r4_r,r5_r,r6_r,r7_r]))))
R_l = np.vstack([R_l, r8_l])
R_r = np.vstack([R_r, r8_r])

p18_l = np.array(list(map(int,eighth_vertex([p11_l, p12_l, p13_l, p14_l, p15_l, p16_l, p17_l]))))
p18_r = np.array(list(map(int,eighth_vertex([p11_r, p12_r, p13_r, p14_r, p15_r, p16_r, p17_r]))))
P1_l = np.vstack([P1_l, p18_l])
P1_r = np.vstack([P1_r, p18_r])

p28_l = np.array(list(map(int,eighth_vertex([p21_l, p22_l, p23_l, p24_l, p25_l, p26_l, p27_l]))))
p28_r = np.array(list(map(int,eighth_vertex([p21_r, p22_r, p23_r, p24_r, p25_r, p26_r, p27_r]))))
P2_l = np.vstack([P2_l, p28_l])
P2_r = np.vstack([P2_r, p28_r])

p38_l = np.array(list(map(int,eighth_vertex([p31_l, p32_l, p33_l, p34_l, p35_l, p36_l, p37_l]))))
p38_r = np.array(list(map(int,eighth_vertex([p31_r, p32_r, p33_r, p34_r, p35_r, p36_r, p37_r]))))
P3_l = np.vstack([P3_l, p38_l])
P3_r = np.vstack([P3_r, p38_r])

# p7_l = np.array(list(map(int, missing_vertex([p18_l, p11_l, p1_l, p2_l, p24_l, p27_l]))))
# p8_l = np.array(list(map(int,eighth_vertex([p2_l, p24_l, p27_l, p18_l, p11_l, p1_l, p7_l]))))

# p9_l = np.array(list(map(int, missing_vertex([p18_l, p11_l, p1_l, p2_l, p24_l, p27_l]))))
# p1_0_l = np.array(list(map(int,eighth_vertex([p2_l, p24_l, p27_l, p18_l, p11_l, p1_l, p7_l]))))

In [17]:
def draw_and_label_points(
    image, 
    points, 
    labels, 
    circle_color=(255,255,255),    
    text_color=(255,255,255),      
    radius=10, 
    thickness=4,
    font_scale=4
):
    for (pt, lbl) in zip(points, labels):
        x, y, _ = pt  # each point is [x, y, 1]
        x, y = int(x), int(y)

        # Draw the circle
        cv2.circle(image, (x, y), radius, circle_color, -1)

        # Put the text slightly above and to the right
        cv2.putText(
            image, lbl, (x + 7, y - 7), 
            cv2.FONT_HERSHEY_PLAIN, 
            font_scale, 
            text_color, 
            thickness
        )

In [33]:

labels_B_l = [f"b{i+1}_l" for i in range(len(B_l))]
labels_A_l = [f"a{i+1}_l" for i in range(len(A_l))]
labels_T_l = [f"t{i+1}_l" for i in range(len(T_l))]
labels_R_l = [f"r{i+1}_l" for i in range(len(R_l))]

labels_P1_l = [f"" for i in range(len(P1_l))]  # p11_l, p12_l, ...
labels_P2_l = [f"" for i in range(len(P2_l))]
labels_P3_l = [f"" for i in range(len(P3_l))]
labels_P_l  = [f"" for i in range(len(P_l))]

labels_S_l = [f"s{i+1}_l" for i in range(len(S_l))]



image_left = cv2.imread("left-camera.jpeg")

left_points_red = [b8_l, a8_l, r8_l, p18_l, p28_l, p38_l] 
left_labels_red = ["b8_l", "a8_l", "r8_l", "p18_l", "p28_l", "p38_l"]

draw_and_label_points(
    image_left, 
    left_points_red, 
    left_labels_red, 
    circle_color=(0,0,255),    # Red circles
    text_color=(0,0,255)       
)

draw_and_label_points(image_left, B_l, labels_B_l)
draw_and_label_points(image_left, A_l, labels_A_l)
draw_and_label_points(image_left, T_l, labels_T_l)
draw_and_label_points(image_left, R_l, labels_R_l)
draw_and_label_points(image_left, P1_l, labels_P1_l)
draw_and_label_points(image_left, P2_l, labels_P2_l)
draw_and_label_points(image_left, P3_l, labels_P3_l)
draw_and_label_points(image_left, P_l,  labels_P_l)
draw_and_label_points(image_left, S_l, labels_S_l)

cv2.imwrite("left_camera_labels.png", image_left)



True

In [34]:
labels_B_r = [f"b{i+1}_r" for i in range(len(B_r))]
labels_A_r = [f"a{i+1}_r" for i in range(len(A_r))]
labels_T_r = [f"t{i+1}_r" for i in range(len(T_r))]
labels_R_r = [f"r{i+1}_r" for i in range(len(R_r))]

labels_P1_r = [f"" for i in range(len(P1_r))]  # p11_r, p12_r, ...
labels_P2_r = [f"" for i in range(len(P2_r))]
labels_P3_r = [f"" for i in range(len(P3_r))]
labels_P_r = [f"" for i in range(len(P_r))]

labels_S_r = [f"s{i+1}_r" for i in range(len(S_r))]


image_right = cv2.imread("right-camera.jpeg")

right_points_red = [b8_r, a8_r, r8_r, p18_r, p28_r, p38_r] 
right_labels_red = ["b8_r", "a8_r", "r8_r", "p18_r", "p28_r", "p38_r"]

draw_and_label_points(
    image_right,
    right_points_red,
    right_labels_red,
    circle_color=(0,0,255),
    text_color=(0,0,255)
)

draw_and_label_points(image_right, B_r, labels_B_r)
draw_and_label_points(image_right, A_r, labels_A_r)
draw_and_label_points(image_right, T_r, labels_T_r)
draw_and_label_points(image_right, R_r, labels_R_r)
draw_and_label_points(image_right, P1_r, labels_P1_r)
draw_and_label_points(image_right, P2_r, labels_P2_r)
draw_and_label_points(image_right, P3_r, labels_P3_r)
draw_and_label_points(image_right, P_r,  labels_P_r)
draw_and_label_points(image_right, S_r, labels_S_r)


cv2.imwrite("right_camera_labels.png", image_right)

True

In [36]:
html_code = f"""
<table>
    <tr>
        <td style="text-align: center;">
            <img src="left_camera_labels.png" width="700"><br>
            <p>Left camera</p>
        </td>
        <td style="text-align: center;">
            <img src="right_camera_labels.png" width="700"><br>
            <p>Right camera</p>
        </td>
    </tr>
</table>
"""

display(HTML(html_code))

Left camera,Right camera


## Fundamental Matrix

In [84]:
def fundamental_matrix(left, right):
    n_9 = []
    for l , r in zip(left,right):
        x1, y1, _ = l
        x2, y2, _ = r
        n_9.append([x1*x2, x1*y2, x1, y1*x2, y1*y2, y1, x2, y2, 1])
    
    n_9 = np.array(n_9)
    U, S, Vh = np.linalg.svd(n_9)
    F_vector = Vh[-1, :]/Vh[-1,-1]
    F = F_vector.reshape((3,3))
    
    return F
        

In [98]:
left = np.array([b1_l, b2_l, b4_l, b5_l, b6_l, b7_l, b8_l, a1_l, a2_l, a3_l, a4_l, a5_l, a6_l, a7_l, a8_l, t1_l, t2_l, t3_l, t4_l, r1_l, r2_l, r3_l, r4_l, r5_l, r6_l, r7_l, r8_l])
right = np.array([b1_r, b2_r, b4_r, b5_r, b6_r, b7_r, b8_r, a1_r, a2_r, a3_r, a4_r, a5_r, a6_r, a7_r, a8_r, t1_r, t2_r, t3_r, t4_r, r1_r, r2_r, r3_r, r4_r, r5_r, r6_r, r7_r, r8_r])

#transform
left[:, 0] = 4000 - left[:, 0]
right[:, 0] = 4000 - right[:, 0]

In [99]:
F= fundamental_matrix(left, right)
print(f"Fundamental matrix:\n {F}")
print()

# Check is det of F close to 0
print(f"det(F) = {np.linalg.det(F)}")

Fundamental matrix:
 [[ 8.13522766e-08  1.37800965e-07 -6.11998770e-04]
 [-1.36559108e-07  8.06741692e-08  1.90313149e-04]
 [ 1.94934062e-04 -5.87873198e-04  1.00000000e+00]]

det(F) = 8.837011281282406e-17


## Essential Matrix

In [100]:
# Calibration matrix K1 = K2 = K
K = np.array([[2800, 0, 2000],
            [0, 2800, 1500],
            [0, 0, 1]])

In [101]:
E = np.matmul(np.matmul(K.T, F), K)
print(f"Essential matrix:\n {E}")

Essential matrix:
 [[ 0.63780185  1.08035956 -0.67925976]
 [-1.07062341  0.63248549  0.10697732]
 [ 0.42783987 -0.53552804  0.08018207]]


## Decomposition of Essential Matrix

In [102]:
Q0 = np.array([[0, -1, 0],
            [1, 0, 0],
            [0, 0, 1]])
E0 =  np.array([[0, -1, 0],
            [1, 0, 0],
            [0, 0, 0]])

In [103]:
U, S, Vh = np.linalg.svd(E)
# Check sign for det(U) and det(Vh)
print(f"det(U) = {np.linalg.det(U)}, det(Vh) = {np.linalg.det(Vh)}")

det(U) = 1.0000000000000004, det(Vh) = 1.0000000000000007


In [114]:
Ec = np.matmul(np.matmul(U, E0), U.T)
Aa = np.matmul(np.matmul(U,Q0),Vh)

print(f"Ec: \n{Ec}")
print(f"Aa: \n{Aa}")

#check det(A) = 1
print(f"\ndet(A) = {np.linalg.det(Aa)}")

Ec: 
[[-1.04672095e-18  8.75920708e-01 -4.51136065e-01]
 [-8.75920708e-01 -7.36186070e-18  1.70994635e-01]
 [ 4.51136065e-01 -1.70994635e-01  6.23700763e-18]]
Aa: 
[[-0.76485561  0.61140763  0.2029202 ]
 [-0.21286289 -0.53716658  0.81617489]
 [ 0.60801751  0.58106176  0.54100087]]

det(A) = 1.0000000000000009


In [112]:
print(np.linalg.det(Aa))

1.0000000000000009


In [115]:
C = [Ec[2,1], Ec[0,2], Ec[1,0]]
print(f"Centar of left camera in coordinate system of right camera (WCS): \n {C}")

Centar of left camera in coordinate system of right camera (WCS): 
 [-0.17099463498377254, -0.4511360646881299, -0.8759207075668861]


## Camera Matrix

In [141]:
T2 = np.hstack((K, np.array([0, 0, 0]).reshape(-1,1)))
print(f"Camera matrix for right camera in coordinate system of right camera (WSC):")
print(np.array2string(T2, formatter={'all': lambda x: f"{x:6}"}))

Camera matrix for right camera in coordinate system of right camera (WSC):
[[  2800      0   2000      0]
 [     0   2800   1500      0]
 [     0      0      1      0]]


In [142]:
T1 = np.hstack((np.matmul(K, Aa.T), np.matmul(np.matmul(-K,Aa.T),C).reshape(-1,1)))
print(f"Camera matrix for left camera in coordinate system of right camera (WSC):")
print(np.array2string(T1, formatter={'all': lambda x: f"{x:10.6f}"}))

Camera matrix for left camera in coordinate system of right camera (WSC):
[[-1735.755308 1036.333688 2784.450754 2609.680731]
 [2016.321669 -279.804076 2438.474237 2354.460558]
 [  0.202920   0.816175   0.541001   0.876778]]


## Triangulation

In [143]:
def triangulation(T1, T2, M1, M2):
    x1, y1, _ = M1
    x2, y2, _ = M2
    m = np.array([
        T1[2,:]*y1 - T1[1,:],
        -T1[2,:]*x1 + T1[0,:],
        T2[2,:]*y2 - T2[1,:],
        -T2[2,:]*x2 + T2[0,:]
    ])
    U, S, Vh = np.linalg.svd(m)
    M = Vh[-1,:]/Vh[-1,-1]
    return M
    

In [156]:
B_l[:, 0] = 4000 - B_l[:, 0]
B_r[:, 0] = 4000 - B_r[:, 0]
box_points = list(map(lambda pair: triangulation(T1, T2, pair[0], pair[1]),
                     zip(B_l, B_r)))

A_l[:, 0] = 4000 - A_l[:, 0]
A_r[:, 0] = 4000 - A_r[:, 0]
acotar_points = np.array(list(map(lambda pair: triangulation(T1, T2, pair[0], pair[1]),
                     zip(A_l, A_r))))

T_l[:, 0] = 4000 - T_l[:, 0]
T_r[:, 0] = 4000 - T_r[:, 0]
tetraeder_points = np.array(list(map(lambda pair: triangulation(T1, T2, pair[0], pair[1]),
                     zip(T_l, T_r))))

R_l[:, 0] = 4000 - R_l[:, 0]
R_r[:, 0] = 4000 - R_r[:, 0]
rubic_points = np.array(list(map(lambda pair: triangulation(T1, T2, pair[0], pair[1]),
                     zip(R_l, R_r))))

P1_l[:, 0] = 4000 - P1_l[:, 0]
P1_r[:, 0] = 4000 - P1_r[:, 0]
P1_points = np.array(list(map(lambda pair: triangulation(T1, T2, pair[0], pair[1]),
                     zip(P1_l, P1_r))))
P2_l[:, 0] = 4000 - P2_l[:, 0]
P2_r[:, 0] = 4000 - P2_r[:, 0]
P2_points = np.array(list(map(lambda pair: triangulation(T1, T2, pair[0], pair[1]),
                     zip(P2_l, P2_r))))
P3_l[:, 0] = 4000 - P3_l[:, 0]
P3_r[:, 0] = 4000 - P3_r[:, 0]
P1_points = np.array(list(map(lambda pair: triangulation(T1, T2, pair[0], pair[1]),
                     zip(P3_l, P3_r))))
P_l[:, 0] = 4000 - P_l[:, 0]
P_r[:, 0] = 4000 - P_r[:, 0]
P_points = np.array(list(map(lambda pair: triangulation(T1, T2, pair[0], pair[1]),
                     zip(P_l, P_r))))

S_l[:, 0] = 4000 - S_l[:, 0]
S_r[:, 0] = 4000 - S_r[:, 0]
sphere_points = np.array(list(map(lambda pair: triangulation(T1, T2, pair[0], pair[1]),
                     zip(S_l, S_r))))


In [163]:
print("Box points:")
for i, point in enumerate(box_points):
    print(f"Point {i}: {np.round(point, 4)}")
print("\nTetraeder points:")
for i, point in enumerate(tetraeder_points):
    print(f"Point {i}: {np.round(point, 4)}")

Box points:
Point 0: [-0.2053 -0.1    -0.267   1.    ]
Point 1: [-0.0745 -0.0933 -0.4537  1.    ]
Point 2: [-0.114   0.0317 -0.5304  1.    ]
Point 3: [-0.2369 -0.1041 -0.2941  1.    ]
Point 4: [-0.084  -0.0766 -0.4525  1.    ]
Point 5: [-0.1258  0.0538 -0.5367  1.    ]
Point 6: [-0.2947  0.0061 -0.3993  1.    ]
Point 7: [-0.2732 -0.0065 -0.3907  1.    ]

Tetraeder points:
Point 0: [-0.0529  0.1627 -0.7152  1.    ]
Point 1: [-0.1312  0.0831 -0.6705  1.    ]
Point 2: [-0.1746  0.1817 -0.7055  1.    ]
Point 3: [-0.1485  0.2459 -0.7501  1.    ]
